In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#api fetching lib
from tqdm import tqdm
import requests
import time

In [2]:
df = pd.read_csv('train_data.csv')

In [3]:
df

,QueryID,ResponseID,QueryName,ResponseName,ReleaseDate,RequiredAge,DemoCount,DeveloperCount,DLCCount,Metacritic,...,LegalNotice,Reviews,SupportedLanguages,Website,PCMinReqsText,PCRecReqsText,LinuxMinReqsText,LinuxRecReqsText,MacMinReqsText,MacRecReqsText
0,10,10,Counter-Strike,Counter-Strike,Nov 1 2000,0,0,1,0,88,...,,,English French German Italian Spanish Simplifi...,NaN,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,
1,20,20,Team Fortress Classic,Team Fortress Classic,Apr 1 1999,0,0,1,0,0,...,,,English French German Italian Spanish,NaN,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,
2,30,30,Day of Defeat,Day of Defeat,May 1 2003,0,0,1,0,79,...,,,English French German Italian Spanish,http://www.dayofdefeat.com/,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,
3,40,40,Deathmatch Classic,Deathmatch Classic,Jun 1 2001,0,0,1,0,0,...,,,English French German Italian Spanish,NaN,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,
4,50,50,Half-Life: Opposing Force,Half-Life: Opposing Force,Nov 1 1999,0,0,1,0,0,...,,,English French German Korean,NaN,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11352,567020,567020,TILE,TILE,Dec 9 2016,0,0,1,0,0,...,,,English,NaN,Minimum:OS: Microsoft(r) Windows(r) Vista / 7P...,,,,,
11353,567660,567660,Baseball Riot,Baseball Riot,Jan 17 2017,0,0,1,0,0,...,Copyright (c) 2016 10tons Ltd.,,English**languages with full audio support,http://www.10tons.com/Game/baseball_riot.html,Minimum:OS: Windows XP / Vista / 7 / 8 / 10Pro...,,,,,
11354,567860,567860,Passage 4,Passage 4,Dec 13 2016,0,0,1,0,0,...,2016 copyright by netmin e.K.,,English* French Italian German* Spanish Dutch*...,http://www.libredia.com,Minimum:OS: Windows 2000/XP/Vista/7/8/10Proces...,,,,,
11355,567940,567940,Piximalism,Piximalism,Sep 26 2019,0,0,1,0,0,...,,,English,NaN,Minimum:OS: Microsoft(r) Windows(r) XP / Vista...,Recommended:OS: Microsoft(r) Windows(r) XP / V...,,,,


In [ ]:
#fetching function
def get_top_tags(app_id):
    """
    fetching popular user tags, more specific than the genre
    """
    url = f"https://steamspy.com/api.php?request=appdetails&appid={app_id}"
    try:
        response = requests.get(url, timeout=10) 
        
        if response.status_code == 200:
            data = response.json()
            tags_dict = data.get('tags', {})
            if isinstance(tags_dict, dict) and tags_dict:
                #--> extracting top 10 tage from the dict returned 
                top_tags = list(tags_dict.keys())[:10]
                return ", ".join(top_tags)
            else:
                return "No Tags" 
                
    except requests.exceptions.RequestException as e:
        return "server err"
        
    return "Failed" #if status is not 200

#--> collect tags 
extracted_tags = []

print(f"Starting API extraction for {len(df)} games")

#-->extraction loop
for index, row in tqdm(df.iterrows(), total=len(df), desc="Fetching Tags"):
    app_id = row['QueryID']
    
    #if the appID is null, tags is null
    if pd.isna(app_id):
        extracted_tags.append(np.nan)
        continue
        
    tag_string = get_top_tags(int(app_id))
    extracted_tags.append(tag_string)
    # avoidd api blocking
    time.sleep(1.5)

Starting API extraction for 11357 games


Fetching Tags: 100%|██████████| 11357/11357 [14:08:39<00:00,  4.48s/it]  ]


In [12]:
df['PopularUserTags'] = extracted_tags

In [13]:
df

,QueryID,ResponseID,QueryName,ResponseName,ReleaseDate,RequiredAge,DemoCount,DeveloperCount,DLCCount,Metacritic,...,Reviews,SupportedLanguages,Website,PCMinReqsText,PCRecReqsText,LinuxMinReqsText,LinuxRecReqsText,MacMinReqsText,MacRecReqsText,PopularUserTags
0,10,10,Counter-Strike,Counter-Strike,Nov 1 2000,0,0,1,0,88,...,,English French German Italian Spanish Simplifi...,NaN,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,,"Action, FPS, Multiplayer, Shooter, Classic, Te..."
1,20,20,Team Fortress Classic,Team Fortress Classic,Apr 1 1999,0,0,1,0,0,...,,English French German Italian Spanish,NaN,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,,"Action, FPS, Multiplayer, Classic, Hero Shoote..."
2,30,30,Day of Defeat,Day of Defeat,May 1 2003,0,0,1,0,79,...,,English French German Italian Spanish,http://www.dayofdefeat.com/,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,,"FPS, World War II, Multiplayer, Shooter, Actio..."
3,40,40,Deathmatch Classic,Deathmatch Classic,Jun 1 2001,0,0,1,0,0,...,,English French German Italian Spanish,NaN,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,,"Action, FPS, Classic, Multiplayer, Shooter, Fi..."
4,50,50,Half-Life: Opposing Force,Half-Life: Opposing Force,Nov 1 1999,0,0,1,0,0,...,,English French German Korean,NaN,Minimum: 500 mhz processor 96mb ram 16mb video...,,Minimum: Linux Ubuntu 12.04 Dual-core from Int...,,Minimum: OS X Snow Leopard 10.6.3 1GB RAM 4GB...,,"FPS, Action, Classic, Sci-fi, Singleplayer, Sh..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11352,567020,567020,TILE,TILE,Dec 9 2016,0,0,1,0,0,...,,English,NaN,Minimum:OS: Microsoft(r) Windows(r) Vista / 7P...,,,,,,"Casual, Indie, Puzzle"
11353,567660,567660,Baseball Riot,Baseball Riot,Jan 17 2017,0,0,1,0,0,...,,English**languages with full audio support,http://www.10tons.com/Game/baseball_riot.html,Minimum:OS: Windows XP / Vista / 7 / 8 / 10Pro...,,,,,,"Casual, Indie, Sports, Action, Arcade, 2D, Sin..."
11354,567860,567860,Passage 4,Passage 4,Dec 13 2016,0,0,1,0,0,...,,English* French Italian German* Spanish Dutch*...,http://www.libredia.com,Minimum:OS: Windows 2000/XP/Vista/7/8/10Proces...,,,,,,"Clicker, Puzzle, Match 3, Board Game, Relaxing..."
11355,567940,567940,Piximalism,Piximalism,Sep 26 2019,0,0,1,0,0,...,,English,NaN,Minimum:OS: Microsoft(r) Windows(r) XP / Vista...,Recommended:OS: Microsoft(r) Windows(r) XP / V...,,,,,"Action, Casual, Adventure, Indie, Pixel Graphics"


In [14]:
df.to_csv('dataset_API.csv', index=False)